# 11. MCP (Model Context Protocol)

## 학습 목표

MCP(Model Context Protocol)를 통해 외부 도구와 컨텍스트를 에이전트에 연결하는 방법을 알아봅니다.

이 노트북에서 다루는 내용:
- MCP의 개념과 아키텍처(서버/클라이언트/호스트)를 이해한다
- `langchain-mcp-adapters` 패키지로 MCP 서버에 연결한다
- `ChatOpenAI.bind_tools(mcp_tools)`로 에이전트와 MCP 도구를 통합한다
- Stdio와 SSE 전송 방식의 차이를 안다
- 다중 MCP 서버를 연결하는 방법을 익힌다

### 사전 준비
- 같은 디렉토리에 `math_server.py`(Stdio 서버)와 `get_weather.py`(HTTP 서버)가 준비되어 있어야 합니다
- weather 서버는 별도 터미널에서 `uv run python get_weather.py`로 미리 실행해 두세요

In [5]:
# .env 파일에서 API 키(OPENAI_API_KEY 등)를 로드합니다.
from dotenv import load_dotenv
import os
load_dotenv(override=True)

print("환경 준비 완료.")

환경 준비 완료.


In [6]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

Langfuse tracing ON — 


In [7]:
# OpenTelemetry 컨텍스트 관련 경고 로그를 억제합니다.
# MCP 클라이언트가 내부적으로 OpenTelemetry를 사용하면서 불필요한 경고가 출력될 수 있습니다.
import logging
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

## 11.2 MCP 개념

**MCP(Model Context Protocol)**는 외부 도구와 컨텍스트를 **표준화된 방식**으로 LLM에 제공하기 위한 오픈 프로토콜입니다.

### 아키텍처 구성 요소

| 구성 요소 | 역할 | 예시 |
|----------|------|------|
| **MCP 서버** | 도구, 리소스, 프롬프트를 노출 | 파일 시스템 서버, DB 서버, API 래퍼 |
| **MCP 클라이언트** | 서버에 연결하여 도구를 가져옴 | `MultiServerMCPClient` |
| **호스트** | 클라이언트를 관리하고 LLM과 연결 | LangChain 에이전트, IDE |

### 핵심 리소스 타입

- **Tools**: 에이전트가 호출할 수 있는 실행 가능한 함수
- **Resources**: 파일, DB 레코드 등의 데이터 (LangChain Blob 객체로 변환)
- **Prompts**: 재사용 가능한 프롬프트 템플릿

### 왜 MCP인가?

MCP 이전에는 각 도구마다 개별적으로 연결 코드를 작성해야 했습니다. MCP는 이를 **하나의 표준 프로토콜**로 통합하여:
- 도구 제공자는 한 번만 MCP 서버를 구현하면 됩니다
- LLM 호스트는 MCP 클라이언트 하나로 모든 도구에 접근할 수 있습니다
- 생태계 전체에서 도구를 재사용할 수 있습니다

### 전송 방식 (Transport)

| 전송 방식 | 프로토콜 | 특징 | 사용 예시 |
|----------|---------|------|----------|
| **Stdio** | 표준 입출력 | 로컬 프로세스 간 통신, 설정 간단 | 로컬 CLI 도구, 스크립트 |
| **Streamable HTTP** | HTTP | 원격 서버 연결 가능, 방화벽 친화적 | 웹 API, 클라우드 서비스 |

> 💡 Stdio는 MCP 클라이언트가 서버 프로세스를 직접 실행(subprocess)하므로 별도 서버 시작이 불필요합니다.
> 반면 HTTP 방식은 서버를 미리 실행해 두어야 합니다.

## 다중 MCP 서버 연결

`MultiServerMCPClient`는 이름 그대로 여러 MCP 서버를 동시에 관리할 수 있습니다.

아래 예제에서는 두 개의 MCP 서버를 동시에 연결합니다:
- **math 서버** (Stdio): `math_server.py`를 서브프로세스로 실행하여 사칙연산 도구 제공
- **weather 서버** (HTTP): 별도로 실행 중인 HTTP 서버에서 날씨 정보 도구 제공

> ⚠️ weather 서버는 먼저 별도 터미널에서 `uv run get_weather.py`를 실행해야 합니다.

In [10]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient  
from langchain.agents import create_agent

# MultiServerMCPClient: 여러 MCP 서버를 딕셔너리 형태로 등록합니다.
# 키(math, weather)는 서버 식별용 이름이며, 도구 이름 충돌을 방지합니다.
client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",  # Stdio: 서브프로세스로 서버를 자동 실행
            "command": "python",
            # math_server.py의 경로 (현재 디렉토리 기준 상대 경로)
            "args": ["./math_server.py"],
        },
        "weather": {
            "transport": "http",  # HTTP: 이미 실행 중인 원격 서버에 연결
            # get_weather.py를 별도 터미널에서 미리 실행해야 합니다
            "url": "http://localhost:8000/mcp",
            # 상용 API와 같이 헤더를 통한 인증이 필요한 경우 아래와 같이 작성합니다.
            # "headers": {
            #     "Authorization": "Bearer YOUR_TOKEN",
            #     "X-Custom-Header": "custom-value"
            # },
        }
    }
)

# MCP 서버들로부터 사용 가능한 도구 목록을 가져옵니다.
tools = await client.get_tools()

# 가져온 MCP 도구들을 LangChain 에이전트에 바인딩합니다.
agent = create_agent(
    "openai:gpt-5.4-mini",
    tools  
)

# 수학 도구 테스트: add와 multiply 도구가 자동으로 호출됩니다.
math_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]},
    config=lf_config
)

# 날씨 도구 테스트: get_weather 도구가 자동으로 호출됩니다.
weather_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]},
    config=lf_config
)

print(math_response['messages'][-1].content)
print(weather_response['messages'][-1].content)

(3 + 5) × 12 = **96**
It’s always sunny in New York.


In [11]:
tools

[StructuredTool(name='add', description='두 정수를 더합니다.', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7684445fd300>),
 StructuredTool(name='multiply', description='두 정수를 곱합니다.', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x76842f48fec0>),
 StructuredTool(name='get_weather', description='지정한 위치의 날씨 정보를 반환합니다.', args_schema={'additionalProperties': False, 'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}, m